In [2]:
from qk_api_contracts.schema.characters import Character, CharacterBase, CharacterListQuery
from qk_api_contracts.enums import GameSystem
from httpx import AsyncClient
from uuid import UUID

BASE_URL = "http://localhost:8080/api"
HEADERS = {
    "X-QK-User-ID": "321",
    "X-QK-Guild-ID": "12345",
    "X-Request-ID": "54321"
}
client = AsyncClient(base_url=BASE_URL)

In [3]:
async def create_character(payload: CharacterBase) -> Character:
    resp = await client.post(
        url="/v1/characters",
        json=payload.model_dump(mode="json"),
        headers=HEADERS
    )
    if resp.status_code == 201:   
        return Character.model_validate(resp.json())
    raise ValueError(resp.status_code, resp.json())

async def get_character(character_id: UUID | str) -> tuple[Character, str]:
    resp = await client.get(
        url=f"/v1/characters/{character_id}",
        headers=HEADERS
    )
    if resp.status_code == 200:
        return Character.model_validate(resp.json()), resp.headers["ETag"].split("/")[1]
    raise ValueError(resp.status_code, resp.json())

async def list_characters(query: CharacterListQuery) -> tuple[list[Character], str]:
    resp = await client.get(
        url=f"/v1/characters",
        headers=HEADERS,
        params=query.model_dump(exclude_none=True, by_alias=True),
    )
    if resp.status_code == 200:
        return [Character(**item) for item in resp.json()["items"]], resp.json()['next_page_token']
    raise ValueError(resp.status_code, resp.json())

async def delete_character(character_id: UUID | str) -> None:
    _, version = await get_character(character_id)
    version
    char = Character.model_validate(resp.json())
    resp = await client.delete(
        url=f"/v1/characters/{char_id}",
        headers={**HEADERS, "If-Match": "1"},
    )

async def update_character(character_id: UUID | str, payload: CharacterBase) -> tuple[Character, str]:
    _, version = get_character(character_id)
    resp = await client.patch(
        url=f"/v1/characters/{char_id}",
        headers={**HEADERS, "If-Match": version},
        json=payload.model_dump(exclude_none=True)
    )

    return resp

In [4]:
party = [
    CharacterBase(
        name="Elandiriel the Great",
        system=GameSystem.PATHFINDER_2E,
        class_name="Wizard",
        subclass_name="Evocation",
        level=19,
        notes="Loves cheese",
        race="Orc"
    ),
    CharacterBase(
        name="Molly the Loud",
        system=GameSystem.PATHFINDER_2E,
        class_name="Bard",
        race="Halfling",
        level=10,
        notes="She's real fucking loud"
    ),
    CharacterBase(
        name="Barbara the Barbarian",
        system=GameSystem.PATHFINDER_2E,
        level=10,
        notes="BARBARBARBARA"
    ),
    CharacterBase(
        name="Bob",
        system=GameSystem.PATHFINDER_2E,
        class_name="Druid",
        level=1,
    ),
    
]
for character in party:
    await create_character(character)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [3]:
CharacterListQuery(user_ids=[6, 5]).model_dump(exclude_none=True)

{'user_ids': [6, 5], 'page_size': 20}

In [5]:
await list_characters(CharacterListQuery(user_ids=[1,], page_size=10))

([], None)

In [36]:
await list_characters(CharacterListQuery(page_size=2, next_page_token='eyJjcmVhdGVkX2F0IjoiMjAyNS0xMi0wNFQxODoxMjo1My4xMDM0NDgrMDA6MDAiLCJsYXN0X2lkIjoiMjg2NGI0OTMtZmJjNC00NzJjLWJjZTUtYWFjYTRlOTgyMmVkIn0='))

([Character(name='Barbara the Barbarian', system=<GameSystem.PATHFINDER_2E: 'Pathfinder 2E'>, level=10, class_name='Barbarian', subclass_name='Dragon', race=None, notes=None, character_id=UUID('f6e04331-00c2-409b-833b-1845c79e6046'), user_id=123),
  Character(name='Bob the Great', system=<GameSystem.PATHFINDER_2E: 'Pathfinder 2E'>, level=19, class_name='Wizard', subclass_name='Evocation', race=None, notes=None, character_id=UUID('4f3bf82a-eb50-42bf-82b3-4497909a5e03'), user_id=123)],
 'eyJjcmVhdGVkX2F0IjoiMjAyNS0xMi0wNFQxODoyNjozMC4xMzE3OTErMDA6MDAiLCJsYXN0X2lkIjoiNGYzYmY4MmEtZWI1MC00MmJmLTgyYjMtNDQ5NzkwOWE1ZTAzIn0=')

In [56]:
await list_characters(CharacterListQuery(user_ids=[436,]))

([Character(name='Bob the Great', system=<GameSystem.PATHFINDER_2E: 'Pathfinder 2E'>, level=19, class_name='Wizard', subclass_name='Evocation', race=None, notes=None, character_id=UUID('cfb83959-629e-4e11-aaf3-4814acb7ffff'), user_id=123),
  Character(name='Molly the Loud', system=<GameSystem.PATHFINDER_2E: 'Pathfinder 2E'>, level=10, class_name='Bard', subclass_name='Enigma', race=None, notes=None, character_id=UUID('2864b493-fbc4-472c-bce5-aaca4e9822ed'), user_id=123),
  Character(name='Barbara the Barbarian', system=<GameSystem.PATHFINDER_2E: 'Pathfinder 2E'>, level=10, class_name='Barbarian', subclass_name='Dragon', race=None, notes=None, character_id=UUID('f6e04331-00c2-409b-833b-1845c79e6046'), user_id=123),
  Character(name='Bob the Great', system=<GameSystem.PATHFINDER_2E: 'Pathfinder 2E'>, level=19, class_name='Wizard', subclass_name='Evocation', race=None, notes=None, character_id=UUID('4f3bf82a-eb50-42bf-82b3-4497909a5e03'), user_id=123),
  Character(name='Bob the Great', sy

In [10]:
query = CharacterListQuery(level_min=18)

resp = await client.get(
        url=f"/v1/characters",
        headers=HEADERS,
        # params=query.model_dump(exclude_none=True),
    )

In [9]:
query.model_dump(exclude_none=True)

{'level_min': 18, 'page_size': 20}

In [49]:
char_id = '9977cedd-84f4-4fe9-b231-2d5bf4cab432'
await get_character(char_id)

(Character(name='Barbara the Barbarian', system=<GameSystem.PATHFINDER_2E: 'Pathfinder 2E'>, level=10, class_name='Barbarian', subclass_name='Dragon', race=None, notes=None, character_id=UUID('9977cedd-84f4-4fe9-b231-2d5bf4cab432'), user_id=123),
 '"1"')

In [16]:
char

Character(name='Barbara the Barbarian', system=<GameSystem.PATHFINDER_2E: 'Pathfinder 2E'>, level=10, class_name='Barbarian', subclass_name='Dragon', race=None, notes=None, character_id=UUID('9977cedd-84f4-4fe9-b231-2d5bf4cab432'), user_id=123)

In [19]:
updated_char = CharacterBase(
    name="Barbara level 14",
    level=14,
    system=char.system
)

In [42]:
resp = await client.delete(
        url=f"/v1/characters/{char_id}",
        headers={**HEADERS, "If-Match": "4"},
        # json=updated_char.model_dump(exclude_none=True)
    )

In [55]:
resp.status_code

409

In [22]:
updated_char.model_dump(exclude_none=True)

{'name': 'Barbara level 14',
 'system': <GameSystem.PATHFINDER_2E: 'Pathfinder 2E'>,
 'level': 14}

In [52]:
await get_character('9977cedd-84f4-4fe9-b231-2d5bf4cab432')

ValueError: (403, {'detail': 'Permission denied on character 9977cedd-84f4-4fe9-b231-2d5bf4cab432 [PermissionDeniedException]'})